# Utilisation de l'API

Ce notebook montre comment interroger l'API de prédiction.

**Avant de commencer**, dans un terminal :

```bash
make run_api        # démarre l'API sur http://127.0.0.1:8000
```

L'API n'a besoin d'aucun modèle pour démarrer : `/predict` répondra `503`
tant que `make run_train` n'aura pas produit de modèle (ou tant que
`PUT /model` n'en aura pas chargé un).

La documentation interactive est sur <http://127.0.0.1:8000/docs>.

In [ ]:
import requests

BASE_URL = "http://127.0.0.1:8000"

## 1. Santé du service

In [ ]:
response = requests.get(f"{BASE_URL}/")
print(response.status_code, response.json())

## 2. État du modèle

Permet de vérifier qu'un déploiement a bien chargé un modèle sans provoquer
de prédiction — et d'obtenir la cause de l'échec le cas échéant.

In [ ]:
response = requests.get(f"{BASE_URL}/model")
print(response.status_code, response.json())

## 3. Prédiction simple

Les paramètres sont validés par Pydantic : une valeur aberrante (distance
négative, heure hors 0-23) ou un champ manquant reçoit un `422` explicite
AVANT d'atteindre le modèle.

In [ ]:
params = {
    "distance_km": 5.0,
    "passengers": 2,
    "hour": 14,
    "day_of_week": "monday",
}

response = requests.get(f"{BASE_URL}/predict", params=params)
print(response.status_code, response.json())

### Cas d'erreur : entrée invalide

In [ ]:
response = requests.get(f"{BASE_URL}/predict", params={**params, "distance_km": -1})
print(response.status_code, response.json())

## 4. Prédiction par lot

Le corps attendu est une **liste** d'objets, et la réponse conserve l'ordre
des entrées.

In [ ]:
payload = [
    {"distance_km": 5.0, "passengers": 2, "hour": 14, "day_of_week": "monday"},
    {"distance_km": 12.5, "passengers": 1, "hour": 23, "day_of_week": "saturday"},
]

response = requests.post(f"{BASE_URL}/predict_batch", json=payload)
print(response.status_code, response.json())

## 5. Équivalent en ligne de commande

⚠️ En quotes **simples**, IPython n'interpole PAS les variables `{...}` : la
commande envoyait littéralement `{payload_str}` au serveur. On écrit donc le
JSON dans un fichier et on le passe à `curl` via `--data-binary @fichier`,
ce qui évite toute interpolation dans une chaîne shell.

In [ ]:
import json
from pathlib import Path

# *.json est ignoré par git : ce fichier temporaire ne pollue pas le dépôt.
Path("payload.json").write_text(json.dumps(payload))

In [ ]:
!curl -s -X POST "{BASE_URL}/predict_batch" \
  -H "Content-Type: application/json" \
  --data-binary @payload.json